# Entropy QK-Norm 实验：GitHub / ZIP → Colab GPU

先在菜单 **运行时 → 更改运行时类型 → GPU**。默认 `SOURCE_MODE="upload_zip"`：直接上传交付 ZIP，适合未公开的研究；不需要公开 GitHub 仓库。也可改成 `SOURCE_MODE="github"`，填写自己的公开 GitHub URL。GitHub 路线需把包内 `qknlab/`、`configs/`、`scripts/`、`docs/` 等内容放在仓库根目录。本 notebook 不会为你公开上传任何内容。

不要在 notebook 写账号或访问 token。私有仓库可以先下载 ZIP，使用上传路线。顺序：环境与测试 → GPU smoke → 数据 → pilot → 审查结果 → 冻结协议 → confirmation → 最终测试。默认不会自动启动确认或全部稳健性实验。

**资源**：Colab 会话与 GPU 配额不保证。17.65M 参数主模型 pilot 6 条轨迹保存检查点约 5.5 GB，confirmation 12 条约 11 GB，另加数据与临时文件；免费 15 GB Drive 无法同时保留全部。开始确认前请将 pilot 完整备份到自己的其他存储并核验，或增加容量。不要删除未备份的结果。查看 [Drive 存储用量](https://drive.google.com/drive/quota)。

本 notebook 提供可执行流程；CPU 验证不能冒充你将获得的 GPU 科学结果。完整判读原则见 `docs/EXPERIMENTS.md`。


In [ ]:
SOURCE_MODE = "github"
REPO_URL = "https://github.com/yuanqizhang479/entropy_qkn_colab.git"
REPO_REF = "main"
RUN_TAG = "entropy_qkn_v1"
PILOT_RUN_SEEDS = [11, 22, 33]
PILOT_MAX_STEPS = None

assert SOURCE_MODE in {"github", "upload_zip"}
if SOURCE_MODE == "github":
    assert "YOUR_" not in REPO_URL, "请先填写自己的公开 GitHub 仓库地址"
    assert REPO_URL.startswith("https://github.com/")
    assert "@" not in REPO_URL, "请勿把账号或 token 写入 URL"
assert RUN_TAG and all(c.isalnum() or c in "_-" for c in RUN_TAG)


In [ ]:
from google.colab import drive
from pathlib import Path, PurePosixPath
import os, sys, json, subprocess, shutil, hashlib, math, zipfile

drive.mount("/content/drive")
PERSIST = Path("/content/drive/MyDrive") / RUN_TAG
PERSIST.mkdir(parents=True, exist_ok=True)

def run(*args):
    args = [str(a) for a in args]
    print(" ".join(args), flush=True)
    subprocess.run(args, check=True)

def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

pin_file = PERSIST / "source_identity.json"
pin = json.loads(pin_file.read_text()) if pin_file.exists() else None
if pin:
    assert pin["source_mode"] == SOURCE_MODE, "当前 RUN_TAG 已属于另一条源码路线；新版本请换 RUN_TAG"
if SOURCE_MODE == "github":
    REPO = Path("/content/entropy_qkn_repo")
    if not REPO.exists():
        run("git", "clone", REPO_URL, REPO)
    else:
        assert (REPO / ".git").exists(), "已有目录不是仓库；请使用新的 Colab 会话"
        origin = subprocess.check_output(["git", "-C", str(REPO), "remote", "get-url", "origin"], text=True).strip()
        assert origin.removesuffix(".git") == REPO_URL.removesuffix(".git"), "现有目录属于另一个仓库"
        status = subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True)
        assert not status.strip(), "仓库有本地修改；请先保存，避免覆盖"
    if pin:
        assert pin["repository"].removesuffix(".git") == REPO_URL.removesuffix(".git")
    reference = pin["commit"] if pin else REPO_REF
    run("git", "-C", REPO, "fetch", "origin", reference)
    run("git", "-C", REPO, "checkout", "--detach", "FETCH_HEAD")
    commit = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
    identity = {"source_mode": "github", "repository": REPO_URL, "commit": commit}
else:
    archive_path = PERSIST / "source_archive.zip"
    if not archive_path.exists():
        from google.colab import files
        uploaded = files.upload()
        assert len(uploaded) == 1, "请只上传一个包含实验代码的 ZIP"
        name, content = next(iter(uploaded.items()))
        assert name.lower().endswith(".zip"), "需要 ZIP 文件"
        archive_path.write_bytes(content)
    archive_sha = file_sha256(archive_path)
    if pin:
        assert pin["archive_sha256"] == archive_sha, "存档 ZIP 与记录不一致；保留旧记录并使用新 RUN_TAG"
    extract_dir = Path("/content/entropy_qkn_uploaded") / archive_sha[:16]
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        members = archive.namelist()
        model_paths = [n for n in members if n.endswith("qknlab/model.py")]
        assert len(model_paths) == 1, "ZIP 必须包含唯一一份 qknlab/model.py"
        prefix = model_paths[0][:-len("qknlab/model.py")]
        for name in members:
            path = PurePosixPath(name)
            assert not path.is_absolute() and ".." not in path.parts, "ZIP 包含不安全路径"
        extraction_marker = extract_dir / ".qkn_extracted_archive_sha256"
        if not extraction_marker.exists():
            archive.extractall(extract_dir)
            extraction_marker.write_text(archive_sha + "\n")
        else:
            assert extraction_marker.read_text().strip() == archive_sha
    REPO = extract_dir / prefix
    identity = {"source_mode": "upload_zip", "archive_sha256": archive_sha}
if pin:
    assert pin == identity, "源码身份发生变化；请使用新的 RUN_TAG"
else:
    pin_file.write_text(json.dumps(identity, indent=2) + "\n")
os.chdir(REPO)
assert Path("qknlab").is_dir(), "请将实验包内容置于仓库根目录"
assert sys.version_info >= (3, 11), "请使用 Python 3.11 或更新运行时"
print("Pinned source:", identity)
print("Persistent results:", PERSIST)


## 安装与校验

保留 Colab 预装的 CUDA PyTorch，不用 CPU wheel 替换。其余直接依赖采用仓库锁定版本。安装之后的验证和训练都在新的 Python 子进程执行，避免 notebook 已加载库的版本缓存。若依赖安装失败，先保留完整报错，不要随意升级后继续确认实验。


In [ ]:
run(sys.executable, "-m", "pip", "install", "-r", "requirements.txt")
run(sys.executable, "scripts/check_environment.py", "--require-cuda", "--out", PERSIST / "colab_environment.json")
run(sys.executable, "-m", "pytest", "-q", "tests")


## GPU smoke：只检查流程

这个小数据集是合成 token；它不能作为 WikiText 或自然语言实验结果。重复运行会校验身份并恢复已有进度。


In [ ]:
SMOKE_DATA = Path("/content/qkn_data/smoke")
if not (SMOKE_DATA / "manifest.json").exists():
    run(sys.executable, "-m", "qknlab.data", "prepare", "--synthetic", "--out", SMOKE_DATA,
        "--max-train-tokens", 4096, "--max-eval-tokens", 2048, "--synthetic-vocab-size", 128)
run(sys.executable, "-m", "qknlab.data", "verify", "--data-dir", SMOKE_DATA)
run(sys.executable, "scripts/run_suite.py", "--stage", "smoke", "--data-dir", SMOKE_DATA,
    "--out", PERSIST / "smoke", "--device", "cuda")


## 主数据：WikiText-103 raw

自动下载 Salesforce 维护的固定版本，无需 HF token 或人工搜索。源数据下载约 315 MB；默认只保留前 20M 个训练 token，官方 validation/test 保留，跨划分完全重复文章从训练侧移除并记录；因此是明确去重的训练变体。第一次准备后把派生文件缓存到 Drive，训练时从 Colab 本地读以提高速度。完整来源、许可和限制见 `docs/DATA.md`。

原始入口：[WikiText 作者页面](https://blog.einstein.ai/the-wikitext-long-term-dependency-language-modeling-dataset/)；[锁定的 HF 版本](https://huggingface.co/datasets/Salesforce/wikitext/tree/b08601e04326c79dfdd32d625aee71d232d685c3/wikitext-103-raw-v1)。


In [ ]:
DATA = Path("/content/qkn_data/wikitext103")
DATA_CACHE = PERSIST / "data_cache" / "wikitext103"
if not (DATA / "manifest.json").exists():
    if (DATA_CACHE / "manifest.json").exists():
        shutil.copytree(DATA_CACHE, DATA, dirs_exist_ok=True)
    else:
        run(sys.executable, "-m", "qknlab.data", "prepare", "--dataset", "wikitext103",
            "--out", DATA, "--max-train-tokens", 20000000)
        shutil.copytree(DATA, DATA_CACHE, dirs_exist_ok=True)
run(sys.executable, "-m", "qknlab.data", "verify", "--data-dir", DATA)
print((DATA / "manifest.json").read_text()[:6000])


## Pilot：3 个配对种子

默认计划含 6 次短程训练，每次 500 步；标准轨迹的 4 个计划检查点做真实 AdamW 状态分支诊断。首次可只执行 seed 11，测量实际耗时后继续其余种子。`--max-steps` 仅临时截断执行，不改变原来的学习率日程。

中断后重新执行本格：完成的训练由检查点验证，诊断只有 checkpoint 哈希和协议匹配时才跳过。不要更改已有结果目录中的配置。


In [ ]:
pilot_args = [sys.executable, "scripts/run_suite.py", "--stage", "pilot", "--data-dir", DATA,
              "--out", PERSIST / "pilot", "--device", "cuda", "--run-seeds", *PILOT_RUN_SEEDS]
if PILOT_MAX_STEPS is not None:
    pilot_args += ["--max-steps", PILOT_MAX_STEPS]
run(*pilot_args)


In [ ]:
# 完整 pilot 后生成统计和图；不完整时检查汇总中的缺失提示，不当作完整结果。
run(sys.executable, "-m", "qknlab.summarize", "--runs-root", PERSIST / "pilot", "--out", PERSIST / "summary_pilot",
    "--phase", "pilot", "--expected-seeds", "11,22,33", "--step", 500)
print("请查看 summary_pilot 和每个 run 的 metrics/summary。复核数值误差、真实步长效应、NLL、剪裁率和显存，再冻结确认方案。")


## 确认阶段：先冻结，再执行

检查 pilot 后才能把下格 `FREEZE_CONFIRMATION` 改为 `True`。冻结文件记录代码、数据清单、配置、种子和诊断 SHA256；它是本研究的流程记录，不是外部注册平台证明。之后任何改动都应保留旧记录并建立新版本。

确认计划为 **6 个全新种子 × 2 个分支 × 2500 步**，另加标准轨迹一步诊断。请先保证足够存储；仅本阶段检查点约 11 GB。可用 `CONFIRM_RUN_SEEDS` 分批运行，但不能根据结果删除计划种子。


In [ ]:
FREEZE_CONFIRMATION = False
CONFIRM_PROTOCOL = PERSIST / "protocol_confirmation_v1.json"
confirmation_args = [sys.executable, "scripts/run_suite.py", "--stage", "confirm", "--data-dir", DATA,
                     "--out", PERSIST / "confirmation", "--device", "cuda", "--protocol", CONFIRM_PROTOCOL]
if FREEZE_CONFIRMATION:
    run(*confirmation_args, "--freeze-only")
else:
    print("尚未冻结：先审查 pilot，然后将 FREEZE_CONFIRMATION 改为 True。")


In [ ]:
RUN_CONFIRMATION = False
CONFIRM_RUN_SEEDS = [101, 202, 303, 404, 505, 606]
if RUN_CONFIRMATION:
    assert CONFIRM_PROTOCOL.exists(), "先冻结确认协议"
    run(*confirmation_args, "--run-seeds", *CONFIRM_RUN_SEEDS)
    run(sys.executable, "-m", "qknlab.summarize", "--runs-root", PERSIST / "confirmation", "--out", PERSIST / "summary_confirmation",
        "--phase", "confirm", "--expected-seeds", "101,202,303,404,505,606", "--step", 2500)
else:
    print("确认实验未启动。审查方案和资源后设置 RUN_CONFIRMATION=True。")


## 解释性稳健性实验与 K 控制

三项稳健性配置分别检验学习率、裁剪和模型宽度。使用 pilot 种子，明确归为探索性；不会合并进六种子确认置信区间。每项都是额外训练预算，建议逐个启用；宽 384 的配置更耗显存与磁盘。不要用稳健性中效果最大的配置替换预先指定的主结果。


In [ ]:
RUN_ROBUSTNESS = False
ROBUSTNESS_CONFIGS = ["robustness_lr", "robustness_no_clip", "robustness_width384"]
if RUN_ROBUSTNESS:
    for name in ROBUSTNESS_CONFIGS:
        run(sys.executable, "scripts/run_suite.py", "--stage", "robustness", "--config", f"configs/{name}.json",
            "--seeds", 11, 22, 33, "--data-dir", DATA, "--out", PERSIST / name, "--device", "cuda")
else:
    print("可选稳健性实验未启动。")


In [ ]:
RUN_K_CONTROL = False
if RUN_K_CONTROL:
    run(sys.executable, "scripts/run_suite.py", "--stage", "pilot", "--data-dir", DATA,
        "--out", PERSIST / "pilot_k_control", "--device", "cuda",
        "--arms", "standard", "q_detach", "k_detach", "--diagnostic-arms", "standard", "q_detach", "k_detach")
else:
    print("可选 K 控制未启动；没有运行就不要声称 Q 特异性。")


## 最终测试与独立语料 PG-19

仅在配置冻结、确认训练完成后打开测试集。以下评估对所有确认种子与两个训练分支运行，报告本分词与上下文协议下的 token NLL；不能直接与词级 WikiText 官方榜单比较。

[PG-19 原研究者仓库](https://github.com/google-deepmind/pg19)提供来源说明。脚本自动读取锁定的官方 Google 存储对象版本，准备测试前缀；不需要下载完整训练集。


In [ ]:
RUN_FINAL_TEST = False
INCLUDE_PG19_TEST = True
if RUN_FINAL_TEST:
    assert CONFIRM_PROTOCOL.exists(), "测试前需要冻结确认协议"
    # 校验当前代码与数据仍匹配，不执行训练。
    run(*confirmation_args, "--dry-run")
    eval_sets = [("wikitext103", DATA)]
    if INCLUDE_PG19_TEST:
        pg19 = Path("/content/qkn_data/pg19")
        pg19_cache = PERSIST / "data_cache" / "pg19"
        if not (pg19 / "manifest.json").exists():
            if (pg19_cache / "manifest.json").exists():
                shutil.copytree(pg19_cache, pg19, dirs_exist_ok=True)
            else:
                run(sys.executable, "-m", "qknlab.data", "prepare", "--dataset", "pg19", "--out", pg19,
                    "--max-eval-tokens", 1000000)
        run(sys.executable, "-m", "qknlab.data", "verify", "--data-dir", pg19)
        shutil.copytree(pg19, PERSIST / "data_cache" / "pg19", dirs_exist_ok=True)
        eval_sets.append(("pg19", pg19))
    for seed in [101, 202, 303, 404, 505, 606]:
        for arm in ["standard", "q_detach"]:
            run_dir = PERSIST / "confirmation" / "runs" / f"seed_{seed}" / arm
            status = json.loads((run_dir / "summary.json").read_text())
            assert status.get("run_complete"), f"确认训练未完成: {run_dir}"
            checkpoint = run_dir / "checkpoints" / "step_002500.pt"
            for dataset_name, dataset_dir in eval_sets:
                output = PERSIST / "test" / dataset_name / f"seed{seed}_{arm}.json"
                if output.exists():
                    previous = json.loads(output.read_text())
                    expected = {
                        "checkpoint_sha256": file_sha256(checkpoint),
                        "data_manifest_sha256": file_sha256(dataset_dir / "manifest.json"),
                        "token_file_sha256": file_sha256(dataset_dir / "test.bin"),
                        "seed": seed, "mode": arm, "phase": "confirm", "split": "test",
                        "checkpoint_step": 2500, "context_length": 256,
                        "n_scored_tokens": min(500000, (dataset_dir / "test.bin").stat().st_size // 2 - 1),
                        "protocol": "nonoverlapping_targets_block_reset_gpt2_bpe",
                    }
                    assert all(previous.get(k) == v for k, v in expected.items()), f"已有测试结果身份不一致: {output}"
                    assert isinstance(previous.get("nll_nats_per_token"), (float, int)) and math.isfinite(previous["nll_nats_per_token"]), f"不完整测试结果: {output}"
                    print("Verified completed evaluation:", output)
                else:
                    run(sys.executable, "-m", "qknlab.evaluate", "--checkpoint", checkpoint, "--data-dir", dataset_dir,
                        "--split", "test", "--out", output,
                        "--device", "cuda", "--max-tokens", 500000, "--allow-test")
    for dataset_name, _ in eval_sets:
        run(sys.executable, "-m", "qknlab.summarize", "--runs-root", PERSIST / "confirmation",
            "--evaluations-root", PERSIST / "test" / dataset_name, "--out", PERSIST / f"summary_confirmation_{dataset_name}",
            "--phase", "confirm", "--expected-seeds", "101,202,303,404,505,606", "--step", 2500)
else:
    print("测试集尚未打开。确认训练和配置冻结完成后设置 RUN_FINAL_TEST=True。")


## 需要带回的结果

保留整个 `RUN_TAG` 目录，尤其是冻结协议、数据清单、环境、每个 run 的 manifest/metrics/summary、诊断 JSON、最终测试和汇总图表。检查点较大，可以另存；备份后验证文件大小和 SHA256，再释放 Drive 空间。不要只提供漂亮的图或最好的一个种子。

如果失败，请带回完整错误日志和出错 run 目录。不要为获得显著结果自行增大学习率、换种子或提前停在最好看的检查点。论文的真实结论将由这些结果决定。
